# Homework 1: measurements, calibration and plots
Run this notebook from top to bottom on a Kaggle T4 x2 accelerator. Only physical GPU 0 is exposed to PyTorch.

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
!git clone --depth 1 https://github.com/kostkornilov/hw1.git /kaggle/working/hw1
%cd /kaggle/working/hw1
!git pull
%pip install -q nvidia-ml-py

In [ ]:
import gc
import json
import math
import random
import threading
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pynvml
import torch
from torch.profiler import ProfilerActivity, profile

from calibrate import save_calibration
from equations import bytes_moved, energy, flops, latency, memory
from models import SimpleCNN

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False

device = torch.device("cuda:0")
model = SimpleCNN().to(device).eval()

RESULTS_DIR = Path("results")
FIGURES_DIR = RESULTS_DIR / "figures"
MEASUREMENTS_PATH = RESULTS_DIR / "measurements.csv"
THETA_PATH = RESULTS_DIR / "theta.json"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

## Reproducible measurement grid

In [ ]:
BASE_IMAGE_SIZES = [32, 64, 128, 224, 256, 384, 512]
BASE_BATCH_SIZES = [1, 2, 4, 8, 16, 32, 64, 128, 256]

grid_rng = random.Random(SEED)
image_candidates = [
    value for value in range(32, 513, 16) if value not in BASE_IMAGE_SIZES
]
sampled_image_sizes = sorted(grid_rng.sample(image_candidates, 4))
batch_candidates = [
    value for value in range(1, 257) if value not in BASE_BATCH_SIZES
]
sampled_batch_sizes = sorted(grid_rng.sample(batch_candidates, 3))

image_sizes = sorted(BASE_IMAGE_SIZES + sampled_image_sizes)
batch_sizes = sorted(BASE_BATCH_SIZES + sampled_batch_sizes)
grid = pd.MultiIndex.from_product(
    [image_sizes, batch_sizes], names=["image_size", "batch"]
).to_frame(index=False)
grid["is_validation"] = (
    ~grid["image_size"].isin(BASE_IMAGE_SIZES)
    | ~grid["batch"].isin(BASE_BATCH_SIZES)
)
assert len(grid) == 132

print(f"Image sizes: {image_sizes} (sampled: {sampled_image_sizes})")
print(f"Batch sizes: {batch_sizes} (sampled: {sampled_batch_sizes})")
print(grid["is_validation"].value_counts().rename(index={False: "train", True: "validation"}))

## Measurement helpers

In [ ]:
pynvml.nvmlInit()
nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)

try:
    pynvml.nvmlDeviceGetTotalEnergyConsumption(nvml_handle)
    cumulative_energy_supported = True
except pynvml.NVMLError:
    cumulative_energy_supported = False

def synchronize():
    torch.cuda.synchronize(device)


def warm_up(input_tensor, repeats=3):
    for _ in range(repeats):
        model(input_tensor)
    synchronize()


def timed_forward(input_tensor):
    synchronize()
    started = time.perf_counter()
    model(input_tensor)
    synchronize()
    return time.perf_counter() - started


def measure_latency(input_tensor):
    probe_s = timed_forward(input_tensor)
    repeats = int(np.clip(math.ceil(0.25 / max(probe_s, 1e-9)), 10, 100))
    samples = np.array([timed_forward(input_tensor) for _ in range(repeats)])
    return float(np.median(samples)), repeats, probe_s


def measure_peak_memory(input_tensor):
    torch.cuda.reset_peak_memory_stats(device)
    model(input_tensor)
    synchronize()
    return int(torch.cuda.max_memory_allocated(device))


def measure_profiler_flops(input_tensor):
    activities = [ProfilerActivity.CPU, ProfilerActivity.CUDA]
    with profile(activities=activities, with_flops=True) as prof:
        model(input_tensor)
        synchronize()
    return int(sum(event.flops or 0 for event in prof.key_averages()))


def _energy_repeats(probe_s):
    return int(np.clip(math.ceil(2.0 / max(probe_s, 1e-9)), 10, 10_000))


def measure_energy_counter(input_tensor, repeats):
    synchronize()
    before_mj = pynvml.nvmlDeviceGetTotalEnergyConsumption(nvml_handle)
    for _ in range(repeats):
        model(input_tensor)
    synchronize()
    after_mj = pynvml.nvmlDeviceGetTotalEnergyConsumption(nvml_handle)
    return (after_mj - before_mj) / 1_000.0 / repeats


def measure_energy_polling(input_tensor, repeats, interval_s=0.02):
    samples = []
    stop = threading.Event()

    def read_power():
        return pynvml.nvmlDeviceGetPowerUsage(nvml_handle) / 1_000.0

    def sampler():
        while not stop.wait(interval_s):
            samples.append((time.perf_counter(), read_power()))

    synchronize()
    started = time.perf_counter()
    samples.append((started, read_power()))
    thread = threading.Thread(target=sampler, daemon=True)
    thread.start()
    for _ in range(repeats):
        model(input_tensor)
    synchronize()
    finished = time.perf_counter()
    stop.set()
    thread.join()
    samples.append((finished, read_power()))

    samples.sort()
    timestamps = np.array([sample[0] for sample in samples])
    powers_w = np.array([sample[1] for sample in samples])
    return float(np.trapz(powers_w, timestamps) / repeats)


def measure_energy(input_tensor, probe_s):
    repeats = _energy_repeats(probe_s)
    if cumulative_energy_supported:
        try:
            return measure_energy_counter(input_tensor, repeats), repeats, "nvml_counter"
        except pynvml.NVMLError:
            pass
    return measure_energy_polling(input_tensor, repeats), repeats, "nvml_power_polling"

print(f"NVML cumulative energy supported: {cumulative_energy_supported}")

## Run or resume all 132 configurations

In [ ]:
if MEASUREMENTS_PATH.exists():
    measurements = pd.read_csv(MEASUREMENTS_PATH)
else:
    measurements = pd.DataFrame()

if measurements.empty:
    completed = set()
else:
    completed = set(
        zip(measurements["image_size"].astype(int), measurements["batch"].astype(int))
    )

measurement_order = grid.sample(frac=1, random_state=SEED + 1).to_dict("records")
gpu_total_memory = int(torch.cuda.get_device_properties(0).total_memory)

for index, config in enumerate(measurement_order, start=1):
    image_size = int(config["image_size"])
    batch = int(config["batch"])
    key = (image_size, batch)
    if key in completed:
        continue

    record = {
        "image_size": image_size,
        "batch": batch,
        "is_validation": bool(config["is_validation"]),
        "oom": False,
        "latency_s": np.nan,
        "memory_bytes": np.nan,
        "energy_j": np.nan,
        "profiler_flops": np.nan,
        "latency_repeats": 0,
        "energy_repeats": 0,
        "energy_backend": "",
        "gpu_total_memory_bytes": gpu_total_memory,
    }
    input_tensor = None

    try:
        gc.collect()
        torch.cuda.empty_cache()
        input_tensor = torch.randn(batch, 3, image_size, image_size, device=device)
        with torch.inference_mode():
            warm_up(input_tensor)
            latency_s, latency_repeats, probe_s = measure_latency(input_tensor)
            record["latency_s"] = latency_s
            record["latency_repeats"] = latency_repeats
            record["memory_bytes"] = measure_peak_memory(input_tensor)
            record["profiler_flops"] = measure_profiler_flops(input_tensor)
            energy_j, energy_repeats, backend = measure_energy(input_tensor, probe_s)
            record["energy_j"] = energy_j
            record["energy_repeats"] = energy_repeats
            record["energy_backend"] = backend
    except torch.cuda.OutOfMemoryError:
        record["oom"] = True
    finally:
        del input_tensor
        gc.collect()
        torch.cuda.empty_cache()

    measurements = pd.concat([measurements, pd.DataFrame([record])], ignore_index=True)
    measurements = (
        measurements.drop_duplicates(["image_size", "batch"], keep="last")
        .sort_values(["image_size", "batch"])
        .reset_index(drop=True)
    )
    measurements.to_csv(MEASUREMENTS_PATH, index=False)
    completed.add(key)
    status = "OOM" if record["oom"] else "ok"
    print(f"[{len(completed):3d}/132] S={image_size:3d}, B={batch:3d}: {status}")

assert len(measurements) == 132
measurements.head()

## Calibrate on the base grid and predict all feasible points

In [ ]:
measurements = pd.read_csv(MEASUREMENTS_PATH)
theta = save_calibration(measurements, THETA_PATH)
latency_theta = theta["latency"]
energy_theta = theta["energy"]

measurements["flops_pred"] = flops(measurements["image_size"], measurements["batch"])
measurements["memory_pred_bytes"] = memory(
    measurements["image_size"], measurements["batch"]
)
measurements["latency_pred_s"] = latency(
    measurements["image_size"], measurements["batch"], latency_theta
)
measurements["energy_pred_j"] = energy(
    measurements["image_size"], measurements["batch"], energy_theta
)

print(json.dumps(theta, indent=2))

## Predicted surfaces with measured points

In [ ]:
surface_sizes = np.linspace(32, 512, 60)
surface_batches = np.geomspace(1, 256, 60)
surface_s, surface_b = np.meshgrid(surface_sizes, surface_batches)

def plot_prediction_surface(predict, measured_column, title, unit, filename):
    predicted = np.asarray(predict(surface_s, surface_b), dtype=float)
    figure = plt.figure(figsize=(9, 7))
    axis = figure.add_subplot(111, projection="3d")
    axis.plot_surface(
        np.log2(surface_s),
        np.log2(surface_b),
        np.log10(predicted),
        cmap="viridis",
        alpha=0.55,
        linewidth=0,
    )

    feasible = measurements.loc[~measurements["oom"].astype(bool)].copy()
    feasible = feasible.loc[np.isfinite(feasible[measured_column]) & (feasible[measured_column] > 0)]
    for validation, label, marker, color in (
        (False, "train measured", "o", "tab:blue"),
        (True, "validation measured", "^", "tab:orange"),
    ):
        points = feasible.loc[feasible["is_validation"].astype(bool) == validation]
        axis.scatter(
            np.log2(points["image_size"]),
            np.log2(points["batch"]),
            np.log10(points[measured_column]),
            marker=marker, color=color, s=28, label=label, depthshade=False,
        )

    size_ticks = [32, 64, 128, 256, 512]
    batch_ticks = [1, 4, 16, 64, 256]
    axis.set_xticks(np.log2(size_ticks), labels=size_ticks)
    axis.set_yticks(np.log2(batch_ticks), labels=batch_ticks)
    axis.set_xlabel("Image size S")
    axis.set_ylabel("Batch size B")
    axis.set_zlabel(f"log10({unit})")
    axis.set_title(title)
    axis.legend(loc="upper left")
    figure.tight_layout()
    figure.savefig(FIGURES_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()

plot_prediction_surface(flops, "profiler_flops", "FLOPs: analytical surface and profiler", "FLOPs", "flops_surface.png")
plot_prediction_surface(memory, "memory_bytes", "Memory: analytical surface and measured peak", "bytes", "memory_surface.png")
plot_prediction_surface(lambda s, b: latency(s, b, latency_theta), "latency_s", "Latency: calibrated surface and measurements", "seconds", "latency_surface.png")
plot_prediction_surface(lambda s, b: energy(s, b, energy_theta), "energy_j", "Energy: calibrated surface and measurements", "joules", "energy_surface.png")

## Relative errors and OOM comparison

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(16, 9))
feasible = measurements.loc[~measurements["oom"].astype(bool)].copy()

error_specs = [
    ("profiler_flops", "flops_pred", "FLOPs"),
    ("memory_bytes", "memory_pred_bytes", "Memory"),
    ("latency_s", "latency_pred_s", "Latency"),
    ("energy_j", "energy_pred_j", "Energy"),
]
for axis, (measured_column, predicted_column, title) in zip(axes.flat, error_specs):
    valid = feasible.loc[
        np.isfinite(feasible[measured_column]) & (feasible[measured_column] > 0)
    ].copy()
    valid["relative_error_percent"] = 100 * np.abs(
        valid[predicted_column] - valid[measured_column]
    ) / valid[measured_column]
    for validation, label, marker in (
        (False, "train", "o"), (True, "validation", "^")
    ):
        points = valid.loc[valid["is_validation"].astype(bool) == validation]
        axis.scatter(points[predicted_column], points["relative_error_percent"], label=label, marker=marker, alpha=0.75)
    axis.set_xscale("log")
    axis.set_xlabel("Predicted value")
    axis.set_ylabel("Absolute relative error, %")
    axis.set_title(title)
    axis.grid(alpha=0.25)
    axis.legend()

oom_axis = axes[1, 1]
memory_ratio = measurements["memory_pred_bytes"] / measurements["gpu_total_memory_bytes"]
oom_axis.scatter(
    memory_ratio, measurements["oom"].astype(int),
    c=measurements["oom"].map({False: "tab:blue", True: "tab:red"}), alpha=0.75,
)
oom_axis.axvline(1.0, color="black", linestyle="--", label="analytical memory = GPU capacity")
oom_axis.set_xscale("log")
oom_axis.set_yticks([0, 1], labels=["fit", "OOM"])
oom_axis.set_xlabel("Predicted memory / GPU capacity")
oom_axis.set_title("OOM classification")
oom_axis.grid(alpha=0.25)
oom_axis.legend()

text_axis = axes[1, 2]
text_axis.axis("off")
metric_lines = ["Calibration metrics", ""]
for metric_name in ("latency", "energy"):
    for split_name in ("train", "validation"):
        values = theta["metrics"][metric_name][split_name]
        metric_lines.append(
            f"{metric_name} {split_name}: MAPE={values['mape_percent']:.2f}%, RMSE={values['rmse']:.4g}"
        )
text_axis.text(0.0, 1.0, "\n".join(metric_lines), va="top", family="monospace")

figure.tight_layout()
figure.savefig(FIGURES_DIR / "errors_and_oom.png", dpi=160, bbox_inches="tight")
plt.show()

## Final artifact check

In [ ]:
print(f"Measurements: {MEASUREMENTS_PATH}")
print(f"Parameters: {THETA_PATH}")
print(f"Figures: {sorted(path.name for path in FIGURES_DIR.glob('*.png'))}")